**MERGE IMAGES IN SUBFOLDERS**

In [1]:
from pathlib import Path
import shutil

img_dir = Path('cafearoundsaigon')

valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
files = [f for f in img_dir.iterdir() if f.suffix.lower() in valid_extensions]

for file_path in files:
    filename = file_path.name

    if '_UTC' in filename:
        foldername = filename.split('_UTC')[0] + '_UTC'

        dest_dir = img_dir / foldername
        dest_dir.mkdir(exist_ok=True)
        shutil.move(str(file_path), str(dest_dir / filename))

**CLIP'S ZERO SHOT VIBES CLASSIFICATION**

In [2]:
import torch
import clip
from PIL import Image
from pathlib import Path
import shutil

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

**ENCODE AND AVERAGE VIBE PROMPTS**

In [3]:
vibe_prompts = {
    "bright and airy": [
        "a photo of a bright airy cafe flooded with natural sunlight, large floor to ceiling windows, white interior, light wood furniture, indoor plants, fresh morning atmosphere",
        "a sunlit minimalist cafe with sheer curtains, soft daylight shadows, open layout, white walls and pale colors, Scandinavian style coffee shop",
        "a luminous coffee shop with glass walls, greenery, bright reflections, clean light tones, spacious and breathable interior",
        "a modern cafe filled with daylight, high ceiling, white and beige palette, soft natural light creating gentle shadows"
    ],

    "industrial and raw": [
        "a photo of an industrial cafe with exposed red brick walls, dark metal pipes, concrete floor, black steel furniture, moody dim lighting",
        "a raw urban coffee shop with unfinished concrete walls, visible ducts, hanging filament bulbs, dark shadows, warehouse style interior",
        "a loft style cafe with rugged textures, iron beams, distressed materials, dark tones, gritty and edgy atmosphere",
        "a grunge cafe interior with worn surfaces, metallic structures, low warm lighting, urban industrial aesthetic"
    ],

    "cozy and warm": [
        "a photo of a cozy warm cafe with soft yellow lighting, wooden furniture, plush chairs, blankets, intimate corner seating",
        "a small coffee shop with warm ambient light, candles on tables, bookshelves, soft textures, relaxing homelike atmosphere",
        "a snug cafe interior with orange warm tones, fairy lights, comfortable sofas, quiet and intimate feeling",
        "a hygge style cafe with soft glow lighting, rustic wood decor, cushions, peaceful and inviting mood"
    ],

    "vintage and retro": [
        "a photo of a vintage retro cafe with antique wooden furniture, patterned floor tiles, warm faded colors, nostalgic atmosphere",
        "a classic coffee shop with old fashioned decor, retro posters, analog clock, worn textures, 1970s style interior",
        "a nostalgic cafe with sepia tones, vintage lighting, ornate details, aged materials, timeless design",
        "a retro cafe with mismatched chairs, decorative wallpaper, old objects, warm dim lighting, historic charm"
    ]
}

text_features = []

with torch.no_grad():
    for vibe, prompts in vibe_prompts.items():
        tokens = clip.tokenize(prompts).to(device)
        embeddings = model.encode_text(tokens)

        embeddings /= embeddings.norm(dim=-1, keepdim=True)
        centroid = embeddings.mean(dim=0, keepdim=True)
        centroid /= centroid.norm(dim=-1, keepdim=True)
        text_features.append(centroid)

text_features = torch.cat(text_features)
vibe_labels = list(vibe_prompts.keys())

**CLASSIFY IMAGES**

In [4]:
img_dir = Path("cafearoundsaigon")

for subfolder in img_dir.iterdir():
    if not subfolder.is_dir() or subfolder.name in vibe_prompts:
        continue

    images = list(subfolder.glob("*.jpg"))
    if not images:
        continue

    images_feature = []

    with torch.no_grad():
        for img_path in images:
            image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
            images_feature.append(model.encode_image(image))

        image_features = torch.cat(images_feature).mean(dim=0, keepdim=True)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # Compare Image Feature Vectors to Vibe Centroids using Cosine Similarity
        similarities = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        score, idx = similarities[0].topk(1)

    best_vibe = vibe_labels[idx[0]]
    final_dir = img_dir / best_vibe
    final_dir.mkdir(exist_ok=True)
    shutil.move(str(subfolder), str(final_dir / subfolder.name))